# Building MCP Servers with FastMCP

**FastMCP** is a Python framework that turns ordinary functions into MCP servers. Instead of hand-writing JSON schemas and JSON-RPC dispatch tables, you decorate functions with `@mcp.tool()`, `@mcp.resource()`, and `@mcp.prompt()`. FastMCP reads type hints and docstrings, registers capabilities, and serves them over stdio or HTTP.

This notebook teaches the FastMCP development model by building a **MiniFastMCP** runtime in plain Python — same decorator API, same lifecycle, no external package required. The scenario is **ShopCo Orders & Policies**: a support team exposes order lookups, policy documents, and reusable prompt templates to any MCP host (Cursor, Claude Desktop, or a custom agent app).

You will learn how to:

1. Register tools, resources, and prompts with decorators
2. Generate JSON schemas automatically from type hints
3. Wire JSON-RPC handlers (`tools/list`, `tools/call`, `resources/read`, …)
4. Validate inputs, return structured errors, and annotate destructive tools
5. Test a server in-process before deploying it as a standalone process

Everything runs offline. A final cell shows how discovered tools would feed a live LLM when you choose to enable it.

In [ ]:
"""
ShopCo FastMCP Lab — decorator-based MCP server construction.
"""

import inspect
import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable, get_args, get_origin


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS: dict[str, dict[str, Any]] = {
    "ORD-1001": {
        "order_id": "ORD-1001",
        "customer": "alice@shop.com",
        "status": "shipped",
        "total_usd": 1299.00,
        "items": ["Laptop Stand", "USB-C Dock"],
        "carrier": "FedEx",
        "tracking": "7489 1234 5678",
    },
    "ORD-1002": {
        "order_id": "ORD-1002",
        "customer": "bob@shop.com",
        "status": "processing",
        "total_usd": 449.00,
        "items": ["Wireless Keyboard"],
        "carrier": None,
        "tracking": None,
    },
}

POLICIES: dict[str, str] = {
    "returns": "# Returns\n\n30-day window. Unused items only. Refund in 5–7 business days.",
    "shipping": "# Shipping\n\nFree over $50. Tracking emailed at carrier scan.",
    "cancel": "# Cancellations\n\nAllowed only before shipment leaves warehouse.",
}

REFUND_QUEUE: list[dict[str, Any]] = []
AUDIT_LOG: list[dict[str, Any]] = []

print(f"ShopCo corpus: {len(ORDERS)} orders, {len(POLICIES)} policies")

---

## 1. Why FastMCP Exists

Hand-rolling an MCP server means repeating the same boilerplate for every capability:

1. Write a JSON Schema for parameters
2. Register the schema in a `tools/list` response
3. Parse `tools/call` arguments and validate them
4. Map protocol errors to JSON-RPC error codes
5. Repeat for resources and prompts

FastMCP collapses that into Python decorators. You write business logic; the framework publishes MCP manifests.

```python
# Real FastMCP (production) — same shape we emulate below
from mcp.server.fastmcp import FastMCP

mcp = FastMCP("shopco-orders")

@mcp.tool()
def get_order(order_id: str) -> dict:
    """Fetch a single order by ID."""
    ...

@mcp.resource("policy://{name}")
def policy_doc(name: str) -> str:
    """Return markdown policy text."""
    ...
```

In this lab, `MiniFastMCP` mirrors that API so you can see exactly what the framework does under the hood.

---

## 2. The Decorator Registration Model

When Python imports your server module, decorators run at **import time** and register callables into internal registries:

| Decorator | Registry | MCP method |
|-----------|----------|------------|
| `@mcp.tool()` | `_tools` | `tools/list`, `tools/call` |
| `@mcp.resource(uri)` | `_resources` | `resources/list`, `resources/read` |
| `@mcp.prompt()` | `_prompts` | `prompts/list`, `prompts/get` |

```
  @mcp.tool()          @mcp.resource("policy://{name}")
       │                        │
       ▼                        ▼
  MiniFastMCP registries (in memory)
       │
       ▼
  JSON-RPC dispatcher → host client
```

The server author never manually constructs `tools/list` payloads — the framework builds them from registered functions.

---

## 3. MiniFastMCP — Schema Generation from Type Hints

FastMCP inspects function signatures and converts annotations to JSON Schema. Our `MiniFastMCP` implements a subset: `str`, `int`, `float`, `bool`, `list[str]`, and optional fields via defaults.

In [ ]:
PYTHON_TO_JSON: dict[Any, dict[str, Any]] = {
    str: {"type": "string"},
    int: {"type": "integer"},
    float: {"type": "number"},
    bool: {"type": "boolean"},
}


def annotation_to_schema(annotation: Any) -> dict[str, Any]:
    origin = get_origin(annotation)
    if origin is list:
        args = get_args(annotation)
        item_type = args[0] if args else str
        return {"type": "array", "items": PYTHON_TO_JSON.get(item_type, {"type": "string"})}
    return PYTHON_TO_JSON.get(annotation, {"type": "string"})


def build_input_schema(fn: Callable) -> dict[str, Any]:
    sig = inspect.signature(fn)
    properties: dict[str, Any] = {}
    required: list[str] = []
    for name, param in sig.parameters.items():
        if param.annotation is inspect.Parameter.empty:
            schema = {"type": "string"}
        else:
            schema = annotation_to_schema(param.annotation)
        if param.default is not inspect.Parameter.empty:
            schema = {**schema, "default": param.default}
        else:
            required.append(name)
        properties[name] = schema
    return {"type": "object", "properties": properties, "required": required}


def first_line(doc: str | None) -> str:
    if not doc:
        return ""
    return doc.strip().split("\n")[0]


def demo_schema():
    def sample(order_id: str, include_items: bool = True) -> dict:
        """Look up an order."""
        return {}
    return build_input_schema(sample)

print(pretty(demo_schema()))


---

## 4. MiniFastMCP Core — Registries and Decorators

In [ ]:
@dataclass
class ToolSpec:
    name: str
    description: str
    input_schema: dict[str, Any]
    handler: Callable
    destructive: bool = False
    read_only: bool = True


@dataclass
class ResourceSpec:
    uri_template: str
    name: str
    description: str
    handler: Callable
    is_template: bool = False


@dataclass
class PromptSpec:
    name: str
    description: str
    arguments: list[dict[str, Any]]
    handler: Callable


PROTOCOL_VERSION = "2024-11-05"


class McpProtocolError(Exception):
    def __init__(self, code: int, message: str, data: Any = None):
        super().__init__(message)
        self.code = code
        self.message = message
        self.data = data


def _match_uri(template: str, uri: str) -> dict[str, str] | None:
    parts = re.split(r"(\{\w+\})", template)
    regex_parts = []
    for part in parts:
        if part.startswith("{") and part.endswith("}"):
            g = part[1:-1]
            regex_parts.append(f"(?P<{g}>[^/]+)")
        else:
            regex_parts.append(re.escape(part))
    m = re.fullmatch("".join(regex_parts), uri)
    return m.groupdict() if m else None


def _tool_result(content: Any) -> dict[str, Any]:
    text = content if isinstance(content, str) else json.dumps(content, default=str)
    return {"content": [{"type": "text", "text": text}], "isError": False}


def _tool_error(message: str) -> dict[str, Any]:
    return {"content": [{"type": "text", "text": message}], "isError": True}


class MiniFastMCP:
    """In-process FastMCP analogue with JSON-RPC dispatch."""

    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self._tools: dict[str, ToolSpec] = {}
        self._resources: list[ResourceSpec] = []
        self._prompts: dict[str, PromptSpec] = {}
        self._initialized = False

    def tool(self, *, destructive: bool = False, read_only: bool = True):
        def decorator(fn: Callable):
            spec = ToolSpec(
                name=fn.__name__,
                description=first_line(fn.__doc__),
                input_schema=build_input_schema(fn),
                handler=fn,
                destructive=destructive,
                read_only=read_only,
            )
            self._tools[spec.name] = spec
            return fn
        return decorator

    def resource(self, uri_template: str, *, name: str | None = None):
        is_template = "{" in uri_template
        def decorator(fn: Callable):
            spec = ResourceSpec(
                uri_template=uri_template,
                name=name or fn.__name__,
                description=first_line(fn.__doc__),
                handler=fn,
                is_template=is_template,
            )
            self._resources.append(spec)
            return fn
        return decorator

    def prompt(self, *, name: str | None = None):
        def decorator(fn: Callable):
            prompt_name = name or fn.__name__
            schema = build_input_schema(fn)
            arguments = [
                {"name": k, "description": "", "required": k in schema.get("required", [])}
                for k in schema.get("properties", {})
            ]
            spec = PromptSpec(
                name=prompt_name,
                description=first_line(fn.__doc__),
                arguments=arguments,
                handler=fn,
            )
            self._prompts[prompt_name] = spec
            return fn
        return decorator


    def capabilities(self) -> dict[str, Any]:
        return {
            "tools": {"listChanged": False},
            "resources": {"subscribe": False, "listChanged": False},
            "prompts": {"listChanged": False},
        }

    def handle(self, method: str, params: dict[str, Any]) -> Any:
        if method == "initialize":
            self._initialized = True
            return {
                "protocolVersion": PROTOCOL_VERSION,
                "capabilities": self.capabilities(),
                "serverInfo": {"name": self.name, "version": self.version},
            }
        if not self._initialized and method != "initialize":
            raise McpProtocolError(-32002, "Server not initialized")

        if method == "tools/list":
            return {"tools": self.list_tools_manifest()}

        if method == "tools/call":
            name = params.get("name", "")
            arguments = params.get("arguments") or {}
            spec = self._tools.get(name)
            if not spec:
                raise McpProtocolError(-32602, f"Unknown tool: {name}")
            try:
                result = spec.handler(**arguments)
                return _tool_result(result)
            except TypeError as e:
                return _tool_error(f"Invalid arguments: {e}")
            except ValueError as e:
                return _tool_error(str(e))

        if method == "resources/list":
            resources = [
                {"uri": r.uri_template, "name": r.name, "description": r.description, "mimeType": "text/plain"}
                for r in self._resources if not r.is_template
            ]
            return {"resources": resources}

        if method == "resources/templates/list":
            templates = [
                {"uriTemplate": r.uri_template, "name": r.name, "description": r.description}
                for r in self._resources if r.is_template
            ]
            return {"resourceTemplates": templates}

        if method == "resources/read":
            uri = params.get("uri", "")
            for spec in self._resources:
                if spec.is_template:
                    groups = _match_uri(spec.uri_template, uri)
                    if groups:
                        body = spec.handler(**groups)
                        return {"contents": [{"uri": uri, "mimeType": "text/plain", "text": body}]}
                elif spec.uri_template == uri:
                    body = spec.handler()
                    return {"contents": [{"uri": uri, "mimeType": "text/plain", "text": body}]}
            raise McpProtocolError(-32602, f"Unknown resource: {uri}")

        if method == "prompts/list":
            prompts = [
                {"name": p.name, "description": p.description, "arguments": p.arguments}
                for p in self._prompts.values()
            ]
            return {"prompts": prompts}

        if method == "prompts/get":
            name = params.get("name", "")
            arguments = params.get("arguments") or {}
            spec = self._prompts.get(name)
            if not spec:
                raise McpProtocolError(-32602, f"Unknown prompt: {name}")
            text = spec.handler(**arguments)
            return {
                "description": spec.description,
                "messages": [{"role": "user", "content": {"type": "text", "text": text}}],
            }

        raise McpProtocolError(-32601, f"Method not found: {method}")

    def list_tools_manifest(self) -> list[dict[str, Any]]:
        return [
            {
                "name": t.name,
                "description": t.description,
                "inputSchema": t.input_schema,
                "annotations": {
                    "destructive": t.destructive,
                    "readOnlyHint": t.read_only,
                },
            }
            for t in self._tools.values()
        ]

print("MiniFastMCP core + JSON-RPC dispatch ready")


---

## 5. Building the ShopCo Server — Tools

Each tool is a plain Python function. Type hints become the model-facing schema; the docstring first line becomes the description. Mark `request_refund` as **destructive** so hosts can gate it behind approval.

In [ ]:
mcp = MiniFastMCP("shopco-support", version="1.2.0")


@mcp.tool(read_only=True)
def get_order(order_id: str) -> dict[str, Any]:
    """Fetch order details by order ID (e.g. ORD-1001)."""
    order_id = order_id.strip().upper()
    if order_id not in ORDERS:
        raise ValueError(f"Order {order_id} not found")
    AUDIT_LOG.append({"action": "get_order", "order_id": order_id, "ts": datetime.now(timezone.utc).isoformat()})
    return ORDERS[order_id]


@mcp.tool(read_only=True)
def search_orders(customer_email: str) -> list[dict[str, Any]]:
    """List orders for a customer email address."""
    email = customer_email.strip().lower()
    return [o for o in ORDERS.values() if o["customer"].lower() == email]


@mcp.tool(destructive=True, read_only=False)
def request_refund(order_id: str, reason: str) -> dict[str, Any]:
    """Submit a refund request for human review (does not auto-refund)."""
    order_id = order_id.strip().upper()
    if order_id not in ORDERS:
        raise ValueError(f"Order {order_id} not found")
    ticket = {
        "ticket_id": f"RF-{uuid.uuid4().hex[:8].upper()}",
        "order_id": order_id,
        "reason": reason.strip(),
        "status": "pending_review",
        "created_at": datetime.now(timezone.utc).isoformat(),
    }
    REFUND_QUEUE.append(ticket)
    return ticket


print(f"Registered {len(mcp._tools)} tools:")
for t in mcp.list_tools_manifest():
    flags = t["annotations"]
    print(f"  • {t['name']}: {t['description']}  [destructive={flags['destructive']}]")

---

## 6. Resources — Static and Templated URIs

Resources expose **read-only context**. Static URIs appear in `resources/list`; templates with `{placeholders}` appear in `resources/templates/list`. The handler receives extracted path parameters.

In [ ]:
@mcp.resource("policy://returns")
def policy_returns() -> str:
    """ShopCo returns policy (markdown)."""
    return POLICIES["returns"]


@mcp.resource("policy://{name}")
def policy_by_name(name: str) -> str:
    """Any named policy document."""
    key = name.strip().lower()
    if key not in POLICIES:
        raise ValueError(f"Unknown policy: {name}")
    return POLICIES[key]


@mcp.resource("order-snapshot://{order_id}")
def order_snapshot(order_id: str) -> str:
    """Frozen JSON snapshot of an order for model context."""
    order = ORDERS.get(order_id.strip().upper())
    if not order:
        raise ValueError(f"Order {order_id} not found")
    return json.dumps(order, indent=2)


static = [r for r in mcp._resources if not r.is_template]
templates = [r for r in mcp._resources if r.is_template]
print(f"Resources: {len(static)} static, {len(templates)} templates")
for r in mcp._resources:
    kind = "template" if r.is_template else "static"
    print(f"  • [{kind}] {r.uri_template}")

---

## 7. Prompts — Reusable Templates

Prompts are parameterized message templates hosts can inject into the agent context. They do not call external APIs — they format instructions.

In [ ]:
@mcp.prompt(name="triage_ticket")
def triage_ticket(customer_message: str, order_id: str = "") -> str:
    """Format a support triage prompt for the agent."""
    order_hint = f" Known order: {order_id}." if order_id else ""
    return (
        "You are ShopCo Support. Cite policy resources before promising refunds.\n"
        f"Customer message: {customer_message}.{order_hint}\n"
        "Steps: 1) verify order 2) check policy 3) propose resolution."
    )


@mcp.prompt(name="escalation_summary")
def escalation_summary(order_id: str, issue: str) -> str:
    """Summarize a case for human escalation."""
    return f"Escalation for {order_id}: {issue}. Include policy citations and refund eligibility."


print(f"Registered {len(mcp._prompts)} prompts: {list(mcp._prompts)}")

---

## 8. JSON-RPC Dispatcher

FastMCP's runtime loop reads JSON-RPC messages from transport (stdio or HTTP) and dispatches to registered handlers. We implement the same method table in-process.

In [ ]:
# Dispatcher is built into MiniFastMCP (see section 4).
print("Protocol methods: initialize, tools/*, resources/*, prompts/*")


---

## 9. In-Process Client — Test Before You Ship

FastMCP servers are usually tested by connecting a real MCP client over stdio. For learning, an in-process client calls `handle()` directly — same protocol, zero transport.

In [ ]:
@dataclass
class InProcessMCPClient:
    server: MiniFastMCP
    request_id: int = 0

    def call(self, method: str, params: dict[str, Any] | None = None) -> Any:
        self.request_id += 1
        try:
            return {"jsonrpc": "2.0", "id": self.request_id, "result": self.server.handle(method, params or {})}
        except McpProtocolError as e:
            return {"jsonrpc": "2.0", "id": self.request_id, "error": {"code": e.code, "message": e.message, "data": e.data}}

    def initialize(self) -> dict[str, Any]:
        return self.call("initialize", {"protocolVersion": PROTOCOL_VERSION, "capabilities": {}, "clientInfo": {"name": "lab-client", "version": "0.1"}})


client = InProcessMCPClient(mcp)
init = client.initialize()
print("Initialize:", init["result"]["serverInfo"])

tools = client.call("tools/list")["result"]["tools"]
print(f"\nDiscovered {len(tools)} tools")
print(pretty(tools[0]))

---

## 10. End-to-End Support Scenario

Simulate what a host does: initialize, read policy resource, call `get_order`, then submit a refund request.

In [ ]:
def run_support_flow():
    c = InProcessMCPClient(mcp)
    c.initialize()

    # 1) Load returns policy into context
    policy = c.call("resources/read", {"uri": "policy://returns"})["result"]["contents"][0]["text"]
    print("Policy excerpt:", policy[:60], "...")

    # 2) Look up order
    order = c.call("tools/call", {"name": "get_order", "arguments": {"order_id": "ORD-1001"}})
    print("\nOrder lookup:", order["result"]["content"][0]["text"][:120], "...")

    # 3) Get triage prompt
    prompt = c.call("prompts/get", {
        "name": "triage_ticket",
        "arguments": {"customer_message": "I want a refund", "order_id": "ORD-1001"},
    })
    print("\nTriage prompt:", prompt["result"]["messages"][0]["content"]["text"][:100], "...")

    # 4) Destructive tool — refund request
    refund = c.call("tools/call", {
        "name": "request_refund",
        "arguments": {"order_id": "ORD-1001", "reason": "Item arrived damaged"},
    })
    print("\nRefund ticket:", refund["result"]["content"][0]["text"])
    print(f"\nRefund queue size: {len(REFUND_QUEUE)}")


run_support_flow()


---

## 11. Validation and Error Surfaces

Good MCP servers return **tool-level errors** (`isError: true`) for business failures (order not found) and reserve JSON-RPC errors for protocol problems (unknown method). Models recover better from tool errors because the content stays in the conversation.

In [ ]:
c = InProcessMCPClient(mcp)
c.initialize()

bad_order = c.call("tools/call", {"name": "get_order", "arguments": {"order_id": "ORD-9999"}})
print("Business error (tool level):")
print(pretty(bad_order["result"]))

bad_tool = c.call("tools/call", {"name": "nonexistent", "arguments": {}})
print("\nProtocol error (JSON-RPC level):")
print(pretty(bad_tool.get("error")))

---

## 12. Transport — stdio vs HTTP/SSE

FastMCP can serve the same registered capabilities over different transports:

| Transport | How it works | Typical use |
|-----------|--------------|-------------|
| **stdio** | Host spawns server as child process; JSON-RPC lines on stdin/stdout | Cursor, Claude Desktop, local agents |
| **HTTP + SSE** | Server listens on a port; client connects via HTTP | Remote shared services, team MCP gateways |

The decorator registrations and dispatcher are **transport-agnostic**. Only the framing layer changes — exactly what `MiniFastMCP.handle()` abstracts away in this lab.

```
  Host spawns:  python shopco_server.py
                     │
              stdin/stdout JSON-RPC
                     │
              FastMCP event loop
                     │
              @mcp.tool handlers
```

---

## 13. Composing Servers — Mounting Sub-Modules

Large teams split MCP servers by domain (orders, billing, catalog). FastMCP supports **mounting** child servers under a parent prefix so one process exposes a unified catalog.

Pattern:

1. `orders_mcp = FastMCP("orders")` with order tools
2. `policies_mcp = FastMCP("policies")` with policy resources
3. `main = FastMCP("shopco")` then `main.mount(orders_mcp, prefix="orders")`

We simulate mounting by merging registries with a name prefix.

In [ ]:
def mount_server(parent: MiniFastMCP, child: MiniFastMCP, prefix: str) -> None:
    for name, spec in child._tools.items():
        parent._tools[f"{prefix}__{name}"] = ToolSpec(
            name=f"{prefix}__{name}",
            description=f"[{prefix}] {spec.description}",
            input_schema=spec.input_schema,
            handler=spec.handler,
            destructive=spec.destructive,
            read_only=spec.read_only,
        )
    parent._resources.extend(child._resources)
    for name, spec in child._prompts.items():
        parent._prompts[f"{prefix}__{name}"] = PromptSpec(
            name=f"{prefix}__{name}",
            description=spec.description,
            arguments=spec.arguments,
            handler=spec.handler,
        )


catalog_mcp = MiniFastMCP("shopco-catalog")
orders_only = MiniFastMCP("orders-slice")

@orders_only.tool(read_only=True)
def count_open_orders() -> dict[str, int]:
    """Return count of non-shipped orders."""
    open_count = sum(1 for o in ORDERS.values() if o["status"] != "shipped")
    return {"open_orders": open_count}

mount_server(catalog_mcp, mcp, prefix="support")
mount_server(catalog_mcp, orders_only, prefix="orders")

cat_client = InProcessMCPClient(catalog_mcp)
cat_client.initialize()
names = [t["name"] for t in cat_client.call("tools/list")["result"]["tools"]]
print(f"Unified catalog: {len(names)} tools")
for n in names:
    print(f"  • {n}")

---

## 14. Security Annotations and Host Policy

MCP tool manifests can include **annotations** (`destructive`, `readOnlyHint`, `openWorldHint`). Hosts use these to decide whether to auto-approve a call or require human confirmation.

| Annotation | `request_refund` | `get_order` |
|------------|------------------|-------------|
| `destructive` | `true` | `false` |
| `readOnlyHint` | `false` | `true` |

FastMCP sets these via decorator kwargs: `@mcp.tool(destructive=True, read_only=False)`.

In [ ]:
class HostApprovalGate:
    def __init__(self, client: InProcessMCPClient):
        self.client = client
        self.approved_destructive = False

    def invoke(self, tool_name: str, arguments: dict[str, Any]) -> dict[str, Any]:
        tools = {t["name"]: t for t in self.client.call("tools/list")["result"]["tools"]}
        meta = tools.get(tool_name, {})
        ann = meta.get("annotations", {})
        if ann.get("destructive") and not self.approved_destructive:
            return {"blocked": True, "reason": "Destructive tool requires user approval"}
        return self.client.call("tools/call", {"name": tool_name, "arguments": arguments})


gate = HostApprovalGate(InProcessMCPClient(mcp))
gate.client.initialize()

blocked = gate.invoke("request_refund", {"order_id": "ORD-1002", "reason": "test"})
print("Without approval:", blocked)

gate.approved_destructive = True
allowed = gate.invoke("request_refund", {"order_id": "ORD-1002", "reason": "test"})
print("\nWith approval:", allowed["result"]["content"][0]["text"][:80], "...")

---

## 15. Offline Agent Loop — Tools from Discovery

A minimal agent loop: discover tools once, match user intent to a tool name with simple rules (stand-in for LLM tool choice), execute, format reply.

In [ ]:
def offline_agent(user_message: str) -> str:
    c = InProcessMCPClient(mcp)
    c.initialize()
    tools = c.call("tools/list")["result"]["tools"]
    tool_names = {t["name"] for t in tools}

    if "ORD-" in user_message.upper():
        oid = re.search(r"ORD-\d+", user_message.upper()).group()
        if "refund" in user_message.lower() and "request_refund" in tool_names:
            if "approve" not in user_message.lower():
                return "Refund requests require supervisor approval. Reply 'approve refund' to continue."
            res = c.call("tools/call", {"name": "request_refund", "arguments": {"order_id": oid, "reason": user_message}})
            return res["result"]["content"][0]["text"]
        res = c.call("tools/call", {"name": "get_order", "arguments": {"order_id": oid}})
        return f"Order info: {res['result']['content'][0]['text']}"

    if "policy" in user_message.lower() or "return" in user_message.lower():
        pol = c.call("resources/read", {"uri": "policy://returns"})["result"]["contents"][0]["text"]
        return f"Returns policy: {pol}"

    return f"Available tools: {', '.join(sorted(tool_names))}"


print(offline_agent("What is the status of ORD-1001?"))
print()
print(offline_agent("I need a refund for ORD-1001, item was damaged"))
print()
print(offline_agent("approve refund for ORD-1001, item was damaged"))

---

## 16. Production FastMCP Checklist

### Server structure
- One module per domain (`orders_server.py`, `policies_server.py`)
- Keep handlers thin — delegate to a service layer
- Use precise type hints; avoid `dict` without structure for critical fields

### Schemas and docs
- First line of docstring = tool description the model sees
- Use enums or `Literal` types for constrained string fields
- Return JSON-serializable dicts, not custom objects

### Operations
- Log every `tools/call` with correlation ID
- Mark mutating tools with `destructive=True`
- Test with in-process client before wiring stdio transport

### Deployment
- stdio for desktop hosts; HTTP/SSE for shared team servers
- Version `serverInfo` semver on breaking schema changes
- Document required environment variables in server README (not in MCP manifest)

---

## 17. Anti-Patterns

| Anti-pattern | Problem | Better approach |
|--------------|---------|------------------|
| God-server with 40 tools | Model confusion, schema bloat | Split + mount by domain |
| Missing type hints | Empty or vague schemas | Annotate every parameter |
| Side effects in resources | Breaks read-only contract | Resources return data only |
| Raising uncaught exceptions | Opaque JSON-RPC -32603 | Catch `ValueError` → `isError` tool result |
| Business logic in decorators | Hard to test | Handler calls service functions |
| Skipping `destructive` flag | Host auto-approves refunds | Annotate mutating tools |

---

## 18. Check Your Understanding

1. What problem do `@mcp.tool()` decorators solve compared to hand-written `tools/list`?
2. Where does the tool **description** shown to the model come from in FastMCP?
3. What is the difference between a static resource URI and a URI **template**?
4. When should you return a tool-level `isError` vs a JSON-RPC error?
5. Why mark `request_refund` with `destructive=True`?
6. What changes between stdio and HTTP transport — the handlers or the framing?
7. How does mounting help large teams ship MCP servers?

---

## 19. Optional — Live LLM with Discovered Tools

Set `USE_LIVE_LLM = True` and provide a valid API key in the environment. The pattern: discover tools from MCP, convert manifests to provider tool format, let the model choose `get_order` or read policy context.

In [ ]:
USE_LIVE_LLM = False

if USE_LIVE_LLM:
    import os
    from openai import OpenAI

    c = InProcessMCPClient(mcp)
    c.initialize()
    discovered = c.call("tools/list")["result"]["tools"]
    openai_tools = [
        {
            "type": "function",
            "function": {
                "name": t["name"],
                "description": t["description"],
                "parameters": t["inputSchema"],
            },
        }
        for t in discovered
    ]
    policy_text = c.call("resources/read", {"uri": "policy://returns"})["result"]["contents"][0]["text"]

    client_llm = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
    response = client_llm.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": f"ShopCo support agent. Policy:\n{policy_text}"},
            {"role": "user", "content": "Can I return order ORD-1001?"},
        ],
        tools=openai_tools,
    )
    print(response.choices[0].message)
else:
    print("Offline mode — set USE_LIVE_LLM = True to call a live model with MCP-discovered tools.")

---

## 20. Summary

FastMCP turns Python functions into MCP servers through decorators:

- **`@mcp.tool()`** — actions with auto-generated JSON Schema from type hints
- **`@mcp.resource(uri)`** — read-only context via stable or templated URIs
- **`@mcp.prompt()`** — parameterized templates for agent instructions

The framework handles `tools/list`, validation, and JSON-RPC dispatch; you focus on ShopCo business logic. Test in-process first, annotate destructive tools for host policy, split large surface areas with mounting, then expose the same server over stdio or HTTP for production hosts.